# LangChain: Evaluation

## Outline:

* Example generation
* Manual evaluation (and debugging)
* LLM-assisted evaluation
* LangChain evaluation platform


## Why do we need evaluation?

When you build a more complex LLM app - chains, retrieval, multiple steps - how do you actually know if it's doing a good job? And if you change something (swap the LLM, change how you retrieve chunks, tweak a parameter), how do you know if that made things better or worse?

That's what evaluation is for. A few ways to approach it:
* look at what's going in and coming out of each step - basically debugging/visualizing the chain
* eyeball a bunch of examples yourself and judge them
* use an LLM to evaluate another LLM's (or chain's) output - this turns out to be really useful, since a lot of these tasks are open-ended and there's no single "correct" string to match against, so plain exact-string matching doesn't really work here


In [2]:
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

Note: LLM's do not always produce the same results. When executing the code in your notebook, you may get slightly different answers that those in the video.

In [3]:
# account for deprecation of LLM model
import datetime
# Get the current date
current_date = datetime.datetime.now().date()

# Define the date after which the model should be set to "gpt-3.5-turbo"
target_date = datetime.date(2024, 6, 12)

# Set the model variable based on the current date
if current_date > target_date:
    llm_model = "gpt-3.5-turbo"
else:
    llm_model = "gpt-3.5-turbo-0301"

## Create our QandA application

In [4]:
from langchain.chains import RetrievalQA
from langchain.chat_models import ChatOpenAI
from langchain.document_loaders import CSVLoader
from langchain.indexes import VectorstoreIndexCreator
from langchain.vectorstores import DocArrayInMemorySearch

In [5]:
file = 'OutdoorClothingCatalog_1000.csv'
loader = CSVLoader(file_path=file)
data = loader.load()

In [6]:
index = VectorstoreIndexCreator(
    vectorstore_cls=DocArrayInMemorySearch
).from_loaders([loader])

In [7]:
llm = ChatOpenAI(temperature = 0.0, model=llm_model)
qa = RetrievalQA.from_chain_type(
    llm=llm, 
    chain_type="stuff", #stuffs the docs into a single prompt
    retriever=index.vectorstore.as_retriever(), #universal adapter
    verbose=True,
    chain_type_kwargs = {
        "document_separator": "<<<<>>>>>" #separates each doc within the combined prompt
    }
)


### Coming up with test datapoints

In [8]:
data[10]

Document(page_content=": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded hem.\n- Pull-on pants have a wide elastic waistband and drawstring, side pockets and a modern slim leg.\n\nImported.", metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 10})

In [9]:
data[11]

Document(page_content=': 11\nname: Ultra-Lofty 850 Stretch Down Hooded Jacket\ndescription: This technical stretch down jacket from our DownTek collection is sure to keep you warm and comfortable with its full-stretch construction providing exceptional range of motion. With a slightly fitted style that falls at the hip and best with a midweight layer, this jacket is suitable for light activity up to 20° and moderate activity up to -30°. The soft and durable 100% polyester shell offers complete windproof protection and is insulated with warm, lofty goose down. Other features include welded baffles for a no-stitch construction and excellent stretch, an adjustable hood, an interior media port and mesh stash pocket and a hem drawcord. Machine wash and dry. Imported.', metadata={'source': 'OutdoorClothingCatalog_1000.csv', 'row': 11})

### Hard-coded examples

In [11]:
examples = [
    {
        "query": "Do the Cozy Comfort Pullover Set\
        have side pockets?",
        "answer": "Yes"
    },
    {
        "query": "What collection is the Ultra-Lofty \
        850 Stretch Down Hooded Jacket from?",
        "answer": "The DownTek collection"
    }
]

### LLM-Generated examples

In [12]:
from langchain.evaluation.qa import QAGenerateChain


In [13]:
example_gen_chain = QAGenerateChain.from_llm(ChatOpenAI(model=llm_model))

In [15]:
new_examples = example_gen_chain.apply_and_parse(
    [{"doc": t} for t in data[:5]]
)

In [20]:
new_examples[4]

{'query': ' What technology sets the EcoFlex 3L Storm Pants apart from other waterproof pants?\n\n',
 'answer': ' The EcoFlex 3L Storm Pants feature TEK O2 technology, which offers the most breathability ever tested in waterproof pants.'}

In [21]:
data[4]

Document(page_content=": 4\nname: EcoFlex 3L Storm Pants\ndescription: Our new TEK O2 technology makes our four-season waterproof pants even more breathable. It's guaranteed to keep you dry and comfortable – whatever the activity and whatever the weather. Size & Fit: Slightly Fitted through hip and thigh. \n\nWhy We Love It: Our state-of-the-art TEK O2 technology offers the most breathability we've ever tested. Great as ski pants, they're ideal for a variety of outdoor activities year-round. Plus, they're loaded with features outdoor enthusiasts appreciate, including weather-blocking gaiters and handy side zips. Air In. Water Out. See how our air-permeable TEK O2 technology keeps you dry and comfortable. \n\nFabric & Care: 100% nylon, exclusive of trim. Machine wash and dry. \n\nAdditional Features: Three-layer shell delivers waterproof protection. Brand new TEK O2 technology provides enhanced breathability. Interior gaiters keep out rain and snow. Full side zips for easy on/off over b

### Combine examples

In [22]:
examples += new_examples

In [25]:
qa.run(examples[0]["query"])



> Entering new RetrievalQA chain...

> Finished chain.


'Yes, the Cozy Comfort Pullover Set does have side pockets.'

## Manual Evaluation

In [26]:
import langchain
langchain.debug = True #shows all the intermediate step results (the full prompt, retrieved docs, token usage, etc.)


In [27]:
qa.run(examples[0]["query"])

[chain/start] [1:chain:RetrievalQA] Entering Chain run with input:
{
  "query": "Do the Cozy Comfort Pullover Set        have side pockets?"
}
[chain/start] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain] Entering Chain run with input:
[inputs]
[chain/start] [1:chain:RetrievalQA > 2:chain:StuffDocumentsChain > 3:chain:LLMChain] Entering Chain run with input:
{
  "question": "Do the Cozy Comfort Pullover Set        have side pockets?",
  "context": ": 10\nname: Cozy Comfort Pullover Set, Stripe\ndescription: Perfect for lounging, this striped knit set lives up to its name. We used ultrasoft fabric and an easy design that's as comfortable at bedtime as it is when we have to make a quick run out.\n\nSize & Fit\n- Pants are Favorite Fit: Sits lower on the waist.\n- Relaxed Fit: Our most generous fit sits farthest from the body.\n\nFabric & Care\n- In the softest blend of 63% polyester, 35% rayon and 2% spandex.\n\nAdditional Features\n- Relaxed fit top with raglan sleeves and rounded h

'Yes, the Cozy Comfort Pullover Set does have side pockets.'

In [28]:
# Turn off the debug mode
langchain.debug = False

## LLM assisted evaluation

In [29]:
predictions = qa.apply(examples)



> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


> Entering new RetrievalQA chain...

> Finished chain.


In [30]:
from langchain.evaluation.qa import QAEvalChain

In [31]:
llm = ChatOpenAI(temperature=0, model=llm_model)
eval_chain = QAEvalChain.from_llm(llm)

In [32]:
graded_outputs = eval_chain.evaluate(examples, predictions)

In [33]:
for i, eg in enumerate(examples):
    print(f"Example {i}:")
    print("Question: " + predictions[i]['query']) #the question itself - generated earlier, either by us or by QAGenerateChain
    print("Real Answer: " + predictions[i]['answer']) #NOT actually "real"/ground truth in the human-verified sense - this is the answer QAGenerateChain generated when it had the WHOLE document as context, so it's treated as the ground truth to compare against
    print("Predicted Answer: " + predictions[i]['result']) #this is the actual thing being evaluated - the answer our qa chain produced by retrieving relevant chunks via the vector db + embeddings, then passing those chunks to the llm
    print("Predicted Grade: " + graded_outputs[i]['text']) #verdict from QAEvalChain - it compares 'answer' (ground truth) vs 'result' (our chain's prediction) and grades it CORRECT/INCORRECT, since plain string matching wouldn't work here
    print()


Example 0:
Question: Do the Cozy Comfort Pullover Set        have side pockets?
Real Answer: Yes
Predicted Answer: Yes, the Cozy Comfort Pullover Set does have side pockets.
Predicted Grade: CORRECT

Example 1:
Question: What collection is the Ultra-Lofty         850 Stretch Down Hooded Jacket from?
Real Answer: The DownTek collection
Predicted Answer: The Ultra-Lofty 850 Stretch Down Hooded Jacket is from the DownTek collection.
Predicted Grade: CORRECT

Example 2:
Question:  What is the approximate weight of the Women's Campside Oxfords per pair?


Real Answer:  The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.
Predicted Answer: The approximate weight of the Women's Campside Oxfords per pair is 1 lb. 1 oz.
Predicted Grade: CORRECT

Example 3:
Question:  What are the dimensions of the small and medium sizes of the Recycled Waterhog Dog Mat, Chevron Weave?


Real Answer:  The small size has dimensions of 18" x 28" and the medium size has dimensions of 22.5"

### Why use an LLM to grade the answers?

Look at the first example: the real answer is `"Yes"`, but the predicted answer is `"The Cozy Comfort Pullover Set, Stripe does have side pockets."`. Both are correct - but as strings they're nothing alike. The word "yes" doesn't even appear in the predicted answer.

If we tried to grade this with exact string matching, or even a regex, it would come back wrong - even though the answer is actually right. There's no single "correct" string a language model has to produce; there are countless valid ways to phrase the same correct answer. As long as two answers carry the same meaning, they should be graded as correct.

That's exactly what an LLM is good at - comparing meaning instead of comparing raw text. This is why `QAEvalChain` uses a language model to compare `answer` (ground truth) against `result` (our chain's prediction), instead of any kind of string-matching approach.


In [34]:
graded_outputs[0]

{'text': 'CORRECT'}

## LangChain evaluation platform

Everything we just did in this notebook - running examples, checking the prompt/context via debug mode, grading outputs - can also be done through a persisted UI instead of one-off print statements. It keeps a running "session" log of every chain run, so you can:

* see the inputs and outputs for the whole chain, and drill down step by step (same info as `langchain.debug = True`, just shown in a nicer UI)
* see exactly what got passed to the LLM at each step - system message, human question, response, and output metadata like token usage
* add any of these runs directly into a dataset with one click, which is a much easier way to build up test examples over time instead of writing or generating them all up front

Basically the same evaluation flywheel from earlier (create examples -> run -> grade), just running continuously in the background instead of inside a single notebook run.
